# 2.TCVectorDB外部Embedding模式示例

知识点讲解：TCVectorDB 外部 Embedding 模式

1. 外部 Embedding 模式的定义
   - 与上一个示例（embedding=None，服务端内置向量化）相对
   - 本模式由「客户端」负责把文本转成向量，数据库只负责存储与检索向量
   - 触发方式：构造 TencentVectorDB 时传入一个真实的 Embeddings 实例
   - 维度决定：Collection 的向量维度由「首次写入的向量」决定并固定（本例 1536 维）

2. 两种模式的完整对比
   ┌──────────────┬────────────────────────┬──────────────────────────┐
   │              │ 内置 Embedding          │ 外部 Embedding            │
   ├──────────────┼────────────────────────┼──────────────────────────┤
   │ embedding 参数│ None                   │ OpenAIEmbeddings 等实例   │
   │ 向量化位置    │ 腾讯云服务端            │ 本地客户端                │
   │ 模型选择      │ 受限于平台提供的模型     │ 完全自由（任意 Embeddings）│
   │ API Key       │ 只需数据库密钥          │ 还需嵌入模型的 API Key    │
   │ 额外费用      │ 含在数据库服务内        │ 按嵌入模型 token 计费     │
   │ 网络往返      │ 1 次（客户端->数据库）   │ 2 次（先模型，再数据库）   │
   │ 跨库迁移      │ 困难（向量不可复现）     │ 容易（可重算相同向量）     │
   │ 可控性        │ 低                     │ 高（可裁剪维度、加前缀等） │
   └──────────────┴────────────────────────┴──────────────────────────┘

3. 何时选择外部 Embedding 模式
   - 需要与其他向量库（Faiss/Pinecone）保持向量一致，便于迁移或多库并行
   - 需要使用特定模型（如 bge-large-zh、自训练模型、私有化部署模型）
   - 需要复用 CacheBackedEmbeddings 做嵌入缓存以降本
   - 需要在向量化前后做自定义处理（如指令前缀、维度裁剪、归一化）

4. 关键约束：维度一致性
   - Collection 创建时的向量维度由「首次写入的向量」决定并固定
   - 本例使用 text-embedding-3-small（1536 维），因此 dataset-external 集合
     被固定为 1536 维；后续若换成 384 维模型写入同一集合，会直接报维度错误
   - 因此 collection_name 命名上区分 builtin / external 非常必要，
     两种模式的向量空间完全不同，混用会导致检索结果毫无意义

5. ⚠ 关键坑：外部 Embedding 必须声明 meta_fields 的 text 字段
   - 课程文档明确指出：LangChain 封装的 TencentVectorDB 存在一个 bug——
     当传递外部 Embedding 模型时，必须配置 meta_fields 属性且字段名为 "text"，
     并在 metadatas 中加上 "text" 字段，数据库才会记录原文信息
   - 若不这样做，写入时只存了 vector 而没存 text，检索时因找不到原文无法构建
     Document 文档，直接报错
   - bug 源码要点（add_texts）：
       if embeddings:
           doc_attrs["vector"] = embeddings[id]   # 外部模式：只存向量
       else:
           doc_attrs["text"] = texts[id]           # 内置模式：存原文
   - 修复写法：
       meta_fields=[MetaField(name="text", data_type=META_FIELD_TYPE_STRING)]
       metadatas = [{"text": text, "page": i} for i, text in enumerate(texts)]
   - 注：本示例未启用 metadatas 故未触发该问题，但生产使用外部模式务必加 text 字段声明

6. meta_fields 元数据字段声明（TCVectorDB 的 metadata 是有模式的）
   - 需要用于过滤的字段必须在构造时通过 meta_fields 预先声明类型，
     声明后服务端才会为该字段建立标量索引，从而支持 expr 表达式过滤
   - 字段类型常量：
     · META_FIELD_TYPE_STRING  -> 字符串
     · META_FIELD_TYPE_UINT64  -> 无符号整数
     · META_FIELD_TYPE_ARRAY   -> 数组
   - 未声明的字段虽可随数据一起存储，但无法参与过滤条件（详见本课第 3 个示例）

7. similarity_search_with_score 与 relevance_scores 的差异
   - 本例使用 similarity_search_with_score，返回「数据库原生分数」
     TCVectorDB 默认使用 cosine 度量，原生分数即余弦相似度，越大越相似
   - 上一个示例使用 similarity_search_with_relevance_scores，
     返回归一化到 [0,1] 的相关性得分，跨数据库语义统一
   - 业务代码做阈值判断时优先用 relevance_scores，调试观察原始数值时用 with_score

8. 最佳实践建议
   - 外部模式下务必用 CacheBackedEmbeddings 包装嵌入模型，显著降低 API 费用
   - 把 database_name 也放进环境变量（本例已这样做），便于多环境切换
   - 反复运行本脚本会重复写入相同文本，调试阶段建议显式传 ids 以实现 upsert 覆盖
   - 生产环境应为写入操作加重试，应对网络抖动与偶发限流


In [ ]:
import os
import dotenv
from langchain_community.vectorstores import TencentVectorDB
from langchain_community.vectorstores.tencentvectordb import (
    ConnectionParams
)
from langchain_openai import OpenAIEmbeddings
dotenv.load_dotenv()
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
db = TencentVectorDB(
    embedding=embedding,
    connection_params=ConnectionParams(
        url=os.environ.get("TC_VECTOR_DB_URL"),
        username=os.environ.get("TC_VECTOR_DB_USERNAME"),
        key=os.environ.get("TC_VECTOR_DB_KEY"),
        timeout=int(os.environ.get("TC_VECTOR_DB_TIMEOUT")),
    ),
    database_name=os.environ.get("TC_VECTOR_DB_DATABASE"),
    collection_name="dataset-external",
)
texts = [
    "笨笨是一只很喜欢睡觉的猫咪",
    "我喜欢在夜晚听音乐，这让我感到放松。",
    "猫咪在窗台上打盹，看起来非常可爱。",
    "学习新技能是每个人都应该追求的目标。",
    "我最喜欢的食物是意大利面，尤其是番茄酱的那种。",
    "昨晚我做了一个奇怪的梦，梦见自己在太空飞行。",
    "我的手机突然关机了，让我有些焦虑。",
    "阅读是我每天都会做的事情，我觉得很充实。",
    "他们一起计划了一次周末的野餐，希望天气能好。",
    "我的狗喜欢追逐球，看起来非常开心。",
]

In [ ]:
ids = db.add_texts(texts)
print("添加文档id列表:", ids)
print(db.similarity_search_with_score("我养了一只猫，叫笨笨"))